# 🇺🇸 CDC Open Data - Complete Examples

This notebook demonstrates comprehensive usage of the CDC Open Data accessor for accessing US public health surveillance data.

## Data Source
- **Portal**: https://data.cdc.gov/
- **API**: Socrata Open Data API (SODA)
- **Coverage**: 50 US states + territories
- **License**: Public Domain (US Government)

## Available Datasets
1. COVID-19 Cases and Deaths
2. Influenza Surveillance
3. HIV/AIDS Surveillance
4. Chronic Disease Indicators (CDI)
5. NNDSS (Notifiable Diseases)
6. Behavioral Risk Factors (BRFSS)
7. Hepatitis Surveillance
8. STD/TB Surveillance

In [ ]:
# Setup and imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from epidatasets.sources.cdc_opendata import (
    CDCOpenDataAccessor, 
    get_cdc_datasets, 
    get_cdc_covid,
    get_cdc_influenza
)

# Plotting setup
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ CDC Open Data Accessor loaded successfully!")
print(f"Current date: {datetime.now().strftime('%Y-%m-%d')}")

## 1. Explore Available Datasets

In [ ]:
# Initialize accessor
cdc = CDCOpenDataAccessor()

# Get all available datasets
all_datasets = cdc.get_available_datasets()

print(f"Total available datasets: {len(all_datasets)}")
print("\n" + "="*80)
print("Datasets by Category:")
print("="*80)

for category in all_datasets['category'].unique():
    cat_data = all_datasets[all_datasets['category'] == category]
    print(f"\n📁 {category.upper().replace('_', ' ')} ({len(cat_data)} datasets):")
    for _, row in cat_data.iterrows():
        print(f"   • {row['dataset_key']}: {row['name']}")
        print(f"     Update: {row['update_frequency']}")

## 2. COVID-19 Surveillance Data

In [ ]:
# Fetch COVID-19 data for multiple states
states = ["CA", "NY", "TX", "FL", "IL"]
covid_data = {}

print("Fetching COVID-19 data...")
for state in states:
    try:
        df = cdc.get_covid_cases(
            state=state,
            start_date="2024-01-01",
            limit=500
        )
        covid_data[state] = df
        print(f"  ✅ {state}: {len(df)} records")
    except Exception as e:
        print(f"  ❌ {state}: {e}")

# Combine all data
if covid_data:
    covid_all = pd.concat(covid_data.values(), ignore_index=True)
    print(f"\n📊 Total records: {len(covid_all)}")
    print(f"\nColumns: {list(covid_all.columns)}")

In [ ]:
# Visualize COVID-19 trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Daily new cases
for state in states:
    if state in covid_data and not covid_data[state].empty:
        df = covid_data[state].sort_values('submission_date')
        axes[0].plot(df['submission_date'], df['new_case'], 
                    marker='o', label=state, linewidth=2, markersize=3, alpha=0.8)

axes[0].set_title('COVID-19 Daily New Cases by State (2024)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('New Cases')
axes[0].legend(title='State', loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: Daily new deaths
for state in states:
    if state in covid_data and not covid_data[state].empty:
        df = covid_data[state].sort_values('submission_date')
        axes[1].plot(df['submission_date'], df['new_death'], 
                    marker='s', label=state, linewidth=2, markersize=3, alpha=0.8)

axes[1].set_title('COVID-19 Daily New Deaths by State (2024)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('New Deaths')
axes[1].legend(title='State', loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Influenza Surveillance

In [ ]:
# Get influenza data
flu_states = ["NY", "CA", "TX"]
flu_data = {}

print("Fetching Influenza surveillance data...")
for state in flu_states:
    try:
        df = cdc.get_influenza_data(
            state=state,
            season="2023-24",
            limit=500
        )
        flu_data[state] = df
        print(f"  ✅ {state}: {len(df)} records")
    except Exception as e:
        print(f"  ❌ {state}: {e}")

if flu_data:
    print(f"\n📊 Sample columns: {list(list(flu_data.values())[0].columns) if flu_data else 'N/A'}")

In [ ]:
# Visualize influenza activity
fig, ax = plt.subplots(figsize=(14, 7))

for state in flu_states:
    if state in flu_data and not flu_data[state].empty:
        df = flu_data[state].sort_values('weekendingdate')
        
        # Plot total specimens tested
        if 'total_specimens' in df.columns:
            ax.plot(df['weekendingdate'], df['total_specimens'],
                   marker='o', label=f'{state} - Total', linewidth=2, alpha=0.8)

ax.set_title('Influenza Surveillance - Total Specimens Tested (2023-24 Season)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Week Ending Date')
ax.set_ylabel('Number of Specimens')
ax.legend(title='State')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Notifiable Diseases (NNDSS)

In [ ]:
# List available notifiable diseases
print("Available Notifiable Diseases:")
print("="*80)

diseases = cdc.list_notifiable_diseases()
for i, disease in enumerate(diseases[:20], 1):
    print(f"{i:2d}. {disease}")

print(f"\n... and {len(diseases) - 20} more diseases")
print(f"\nTotal: {len(diseases)} notifiable diseases")

In [ ]:
# Get data for specific diseases
target_diseases = ["Lyme disease", "Measles", "Pertussis"]

print("\nFetching NNDSS data for selected diseases...")
nndss_data = {}

for disease in target_diseases:
    try:
        df = cdc.get_nndss_data(
            disease=disease,
            year=2023,
            limit=1000
        )
        nndss_data[disease] = df
        print(f"  ✅ {disease}: {len(df)} records")
    except Exception as e:
        print(f"  ❌ {disease}: {e}")

# Summarize by state
if nndss_data:
    print("\n" + "="*80)
    print("Top 10 States by Disease (2023):")
    print("="*80)
    
    for disease, df in nndss_data.items():
        if not df.empty and 'state' in df.columns and 'cases' in df.columns:
            state_totals = df.groupby('state')['cases'].sum().sort_values(ascending=False)
            print(f"\n{disease}:")
            for state, cases in state_totals.head(5).items():
                print(f"  {state}: {cases:,} cases")

## 5. Chronic Disease Indicators (CDI)

In [ ]:
# Get diabetes data for all states
print("Fetching Chronic Disease Indicators...")

try:
    diabetes = cdc.get_chronic_disease_indicator(
        indicator="DIABETES",
        year=2021,
        limit=2000
    )
    print(f"  ✅ Diabetes data: {len(diabetes)} records")
    
    # Filter for overall prevalence (no stratification)
    if 'stratification1' in diabetes.columns:
        diabetes_overall = diabetes[diabetes['stratification1'] == 'Overall']
        
        # Get state-level data
        if 'locationabbr' in diabetes_overall.columns:
            state_diabetes = diabetes_overall[
                (diabetes_overall['locationabbr'] != 'US') & 
                (diabetes_overall['datavalue'].notna())
            ].copy()
            
            # Convert to numeric
            state_diabetes['datavalue'] = pd.to_numeric(
                state_diabetes['datavalue'], errors='coerce'
            )
            
            # Sort by prevalence
            top_states = state_diabetes.nlargest(10, 'datavalue')
            
            print(f"\n📊 Top 10 States - Diabetes Prevalence (2021):")
            for _, row in top_states.iterrows():
                print(f"  {row['locationabbr']}: {row['datavalue']:.1f}%")
except Exception as e:
    print(f"  ❌ Error: {e}")

In [ ]:
# Visualize diabetes prevalence by state
if 'top_states' in locals() and not top_states.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    states_list = top_states['locationabbr'].tolist()
    values = top_states['datavalue'].tolist()
    
    bars = ax.barh(states_list, values, color='steelblue', alpha=0.8)
    
    # Add value labels
    for bar, val in zip(bars, values):
        ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, 
                f'{val:.1f}%', va='center', fontsize=10)
    
    ax.set_title('Top 10 States - Diabetes Prevalence (2021)', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Prevalence (%)')
    ax.set_ylabel('State')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## 6. Dataset Summary and Export

In [ ]:
# Get dataset information
print("Dataset Information Summary:")
print("="*80)

dataset_info = cdc.get_dataset_info("covid_cases_deaths")
for key, value in dataset_info.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

# Overall summary
summary = {
    'total_datasets': len(all_datasets),
    'categories': all_datasets['category'].nunique(),
    'states_covered': 50,
    'notifiable_diseases': len(cdc.list_notifiable_diseases())
}

print("\n" + "="*80)
print("CDC Open Data Summary:")
print("="*80)
for key, value in summary.items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

In [ ]:
# Export sample data to CSV
if covid_data:
    output_file = "cdc_covid_sample_data.csv"
    covid_all.to_csv(output_file, index=False)
    print(f"\n✅ Data exported to: {output_file}")
    
    from pathlib import Path
    file_size = Path(output_file).stat().st_size / 1024
    print(f"📁 File size: {file_size:.2f} KB")
    print(f"📊 Records: {len(covid_all)}")

## Summary

This notebook demonstrated:

### ✅ Key Features
1. **15+ datasets** across 6 categories
2. **All 50 US states + territories** coverage
3. **No API key required** - public data
4. **Socrata API integration** with SoQL queries
5. **Built-in caching** for performance

### 📊 Datasets Covered
- COVID-19 Cases and Deaths (daily updates)
- Influenza Surveillance (weekly)
- NNDSS - 60+ notifiable diseases
- Chronic Disease Indicators
- HIV/AIDS, Hepatitis, STD surveillance

### 🔧 Usage Patterns
- Filter by state (2-letter codes)
- Date range queries
- Disease-specific searches
- Multi-state comparisons

### 📚 Resources
- CDC Open Data: https://data.cdc.gov/
- Socrata API Docs: https://dev.socrata.com/
- Accessor Source: `scripts/accessors/cdc_opendata.py`